# 13.16 拿高分也可能學壞嗎？


你想要正確答案，卻把「回答越長越好」寫成獎勵。模型若學得很認真，可能正好給出一大篇錯話。這不是它違反了訓練訊號，而是訊號獎勵的事情與你真正想要的事情不同。先用一個透明反例看見這個差別，再看本章實測究竟出現了什麼。

前置是[13.10的評分員](https://birdhackor.github.io/tiny-perceptron-vlm/13.10.html)、[13.15的示範選卡起點與PPO更新](https://birdhackor.github.io/tiny-perceptron-vlm/13.15.html)、[13.5的DPO偏好代價](https://birdhackor.github.io/tiny-perceptron-vlm/13.5.html)與[7.13的內容、格式分開驗收](https://birdhackor.github.io/tiny-perceptron-vlm/7.13.html)。13.15所需的13.13、13.14分別教裁切與固定參考；本節接著檢查這些方法的結果。下面的`len`只數字串字元個數，不檢查數學；答案與真值都由我們寫好，沒有使用已訓練模型。


In [ ]:
# @title 準備本節的工具（首次執行）
from pathlib import Path
import os
import subprocess
import sys

# Colab 會先下載專案；本機 Jupyter 沿用已開啟的 repo。
try:
    import google.colab
except ImportError:
    in_colab = False
else:
    in_colab = True

if in_colab:
    root = Path("/content/tiny-perceptron-vlm")
    if not root.exists():
        subprocess.run(
            ["git", "clone", "--depth", "1", "https://github.com/birdhackor/tiny-perceptron-vlm.git", str(root)],
            env={**os.environ, "GIT_LFS_SKIP_SMUDGE": "1"},
            check=True,
        )
    subprocess.run([sys.executable, "-m", "pip", "install", "--quiet", "-e", str(root)], check=True)
    os.chdir(root)

root = Path.cwd().resolve()
while not (root / "pyproject.toml").exists() and root != root.parent:
    root = root.parent
if not (root / "tiny_perceptron").is_dir():
    raise RuntimeError("本機請先依 course/first-steps.md 開啟 repo；免安裝練習可使用本節的 Colab 入口")
sys.path.insert(0, str(root))
import torch

torch.set_num_threads(1)
torch.manual_seed(42)

In [ ]:
answers = ["3", "我已經仔細思考過，答案是4。"]
scores = [len(answer) for answer in answers]
chosen = max(range(len(answers)), key=lambda index: scores[index])
print("評分規則挑中", answers[chosen])
print("符合只回數字且算對", answers[chosen] == str(1 + 2))

`scores`保存每篇長度，`max`選出分數最高的編號，再取那篇文字。輸出挑中長句，最後為False：它較長，數字卻錯，也違反只回數字。這是人工設計的評分漏洞。**reward hacking，獎勵鑽漏洞**，指策略利用評分規則的弱點拿到高分，卻沒有完成真正任務；不必假設模型有人的欺騙意圖，數值目標本身就可能導向這個結果。[DeepSeek-R1原論文2.2.2節](https://arxiv.org/pdf/2501.12948v1)把這種漏洞列為學習式獎勵模型可能遇到的問題；它不是只有長度評分才會發生。

獎勵模型不一定只看長度，但它可能從比較資料學到其他捷徑。只看訓練reward上升，不能排除它。應把原始回答交給另一套事先定好的檢查：算式真值、明確格式、資訊是否足夠，或獨立審閱。這套檢查不再讀獎勵模型的分數當答案，否則只是同一位評分員替自己背書。固定參考與PPO裁切也不能把錯誤獎勵自動改成正確標準。

本章實測保留18道最後檢查，每個條件六題，結果如下。「示範選卡起點」是13.15用44筆只回數字示範教出的SFT模型；PPO與DPO分別從這份相同權重開始，前者接評分員與取樣更新，後者直接學偏好對。成功表示選到滿足完整要求的那張卡；所有候選都是程式預寫，不是模型生成的算術答案。

| 最後檢查條件 | 示範選卡起點 | PPO後 | DPO後 |
| --- | ---: | ---: | ---: |
| 只回數字 | 6／6 | 6／6 | 6／6 |
| 用一句話說明 | 0／6 | 0／6 | 0／6 |
| 購買數量缺少，應求補資訊 | 0／6 | 6／6 | 6／6 |
| 完整要求合計 | 6／18 | 12／18 | 12／18 |

這份任務的「一句話說明」只檢查候選是否包含算式與結果的句子，不衡量解釋加法原理的深度。即使在這麼有限的要求上，兩支仍失敗。看`1+2`的說明題：評分員將句子卡排最高，原始分數約13.5565，短答卡約8.7679；PPO卻給短答約0.7928、句子約0.2014，取最高機率時仍選`3`。答對數字與完成要求，確實是兩個結果。

評分員在最後90個偏好對全部排對，但不能把這90／90當成回答策略的90／90。這次說明題在訓練側也0／44通過，不只是新家族泛化失敗；固定短訓配方尚未學好這個條件。它示範的是「評分員排序正確，選擇模型仍未達標」，沒有觀察到策略利用已學評分員的漏洞。上面的長度反例與正式實驗，必須分開報告。

練習只把長度評分改成`int(answer == str(1 + 2))`，先預測短答得1、長句得0，選擇改成`3`，再執行核對。這個規則適合當前可精確檢查的小題，但遇到開放式解釋就不能只用字串相等；若標準不適用，換一個reward公式不會自動解決。
